# Monitorear modelo usando Lakehouse Monitoring
Databricks Lakehouse Monitoring([AWS](https://docs.databricks.com/en/lakehouse-monitoring/index.html)|[Azure](https://learn.microsoft.com/en-us/azure/databricks/lakehouse-monitoring/)) para monitorear tabla de inferencias.

Databricks Lakehouse Monitoring ata un monitor de datos a cualquier tabla Delta y genera pipelines para calcular métricas de calidad. Solo necesita indicar la frecuencia con la cual recolecta datos.

Util para monitorear desvio de datos, etiquetas, prediciones y cambios en las métricas de calidad del modelo.
Crea dos tablas por cada tabla a monitorear:
- Tabla perfil de metricas (con sufijo `_profile_metrics`). Almacena metricas como % de valores nulos, estadísticas descriptivas, metricas de modelo como: accuracy, RMSE, fairness, bias metrics, etc.
- Tabla metricas de desviaciones (drift) (con sufijo `_drift_metrics`). Almacena metricas como "delta" entre % valores nulos, promedio y metricas de pruebas estadisticas para detectar desviación de datos.

## 1. Configuracion

In [0]:
catalog = "medallion_dev"
schema = "gold"
model_name = "customer_churn_classifier"

dbutils.widgets.text("model_full_name", f"{catalog}.{schema}.{model_name}", "Modelo nombre completo: catalog.schema.name") 
dbutils.widgets.text("project_path", "/Workspace/Proyectos_Dev/databricks-medallion", "Ruta del proyecto")

In [0]:
model_full_name = dbutils.widgets.get("model_full_name")
project_path = dbutils.widgets.get("project_path")
catalog = model_full_name.split('.')[0]
schema = model_full_name.split('.')[1]
model_name = model_full_name.split('.')[-1]
feature_table_name = f"{catalog}.{schema}.churn_user_features"
target_col = "churn"

table_offline_inferences = "Churn_Offline_Inference"
table_inferences = "Churn_Inference"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE `{catalog}`.`{schema}`")

## Crear/actualizar tabla de inferencia
Podemos tener una sola tabla como una unión para inferencia offline y online. Pero también puede tener tablas independientes.

In [0]:
%sql
CREATE OR REPLACE TABLE Churn_Inference AS
          SELECT * FROM Churn_Offline_Inference 
          LEFT JOIN churn_user_features USING(user_id) 
          ORDER BY inference_timestamp;

ALTER TABLE Churn_Inference SET TBLPROPERTIES (delta.enableChangeDataFeed = true)

In [0]:
%sql
MERGE INTO Churn_Inference AS i
  USING churn_user_features AS l ON i.user_id == l.user_id
  WHEN MATCHED THEN UPDATE SET i.churn == l.churn

## Crear/actualizar tabla base
En Databricks Lakehouse Monitoring una "tabla línea base" es un punto de referencia para medir las desviaciones de datos y calidad. Es opcional, pero un componente valioso de comparación.

**Referencia para calculos de desvíos:** El principal propósito de la tabla línea base es proveer una distribución estable de los datos. El monitor compara el estado actual con la línea base.
**Refleja la calidad esperad:** La línea base es una instantánea cuando la calidad conocida es alta o un estado curado con distribución ideal.
**Consistente esquema:** Esquema preciso.
**Aplicabilidad en perfil de inferencia:** Representa la distribución esperada permitiendo detectar desvíos.

In [0]:
%sql
CREATE OR REPLACE TABLE churn_user_features_baseline AS
  SELECT * EXCEPT (user_id) 
  FROM Churn_Inference 
  LEFT JOIN churn_user_features USING(user_id);

## Crear metricas personalizadas
Metricas personalizadas puden ser definidas y calculadas automaticamente por Lakehouse monitor. Sirven para capturar algunos aspectos de la lógica del negocio o como una forma de puntuación.
Por ejemplo, calcular el impacto en el negocio (pérdida en cargos mensuales) de un mal desempeño del modelo.

In [0]:
from pyspark.sql.types import DoubleType, StructField
from databricks.sdk.service.catalog import MonitorMetric, MonitorMetricType

expected_loss_metric = [
  MonitorMetric(
    type=MonitorMetricType.CUSTOM_METRIC_TYPE_AGGREGATE,
    name="expected_loss",
    input_columns=[":table"],
    definition="""avg(CASE
    WHEN {{prediction_col}} != {{target_col}} AND {{target_col}} = 'Yes' THEN -monthly_charges
    ELSE 0 END
    )""",
    output_data_type= StructField("output", DoubleType()).json()
  )
]

## Crear monitor
Como estamos monitoreando una tabla de inferencia, tomamos un [perfil de inferencia](https://learn.microsoft.com/en-us/azure/databricks/lakehouse-monitoring/create-monitor-api#inferencelog-profile) del monitor.

In [0]:
import os
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import MonitorInferenceLog, MonitorInferenceLogProblemType, MonitorCronSchedule

print(f"Crear monitor para la tabla de inferencia {catalog}.{schema}.Churn_Inference")
w = WorkspaceClient()

try:
  info = w.quality_monitors.create(
    table_name=f"{catalog}.{schema}.Churn_Inference",
    inference_log=MonitorInferenceLog(
            problem_type=MonitorInferenceLogProblemType.PROBLEM_TYPE_CLASSIFICATION,
            prediction_col="prediction",
            timestamp_col="inference_timestamp",
            granularities=["1 day"],
            model_id_col="model_version",
            target_col=target_col, # optional
    ),
    schedule=MonitorCronSchedule(
        quartz_cron_expression="0 0 12 * * ?",  # Schedules at 12 noon every day
        timezone_id="PST"
    ),
    assets_dir=f"{os.getcwd()}/monitoring", # Change this to another folder of choice if needed
    output_schema_name=f"{catalog}.{schema}",
    baseline_table_name=f"{catalog}.{schema}.churn_user_features_baseline",
    slicing_exprs=["income_bracket='low'", "age"], # Slicing dimension
    custom_metrics=expected_loss_metric)
except Exception as lhm_exception:
  if "already exist" in str(lhm_exception).lower():
    print(f"Monitor for {catalog}.{schema}.Churn_Inference already exists, retrieving monitor info:")
    info = w.quality_monitors.get(table_name=f"{catalog}.{schema}.Churn_Inference")
  else:
    raise lhm_exception

In [0]:
import time
from databricks.sdk.service.catalog import MonitorInfoStatus, MonitorRefreshInfoState

# Wait for monitor to be created
while info.status == MonitorInfoStatus.MONITOR_STATUS_PENDING:
  info = w.quality_monitors.get(table_name=f"{catalog}.{schema}.Churn_Inference")
  time.sleep(10)

assert info.status == MonitorInfoStatus.MONITOR_STATUS_ACTIVE, "Error creating monitor"

Cuando el monitor es creado la primera vez **disparamos un refresh inicial** así que esperamos hasta que termine

In [0]:
def get_refreshes():
  return w.quality_monitors.list_refreshes(table_name=f"{catalog}.{shema}.Churn_Inference").refreshes

refreshes = get_refreshes()
if len(refreshes) == 0:
  # Kick a refresh if none exists
  w.quality_monitors.run_refresh(table_name=f"{catalog}.{shema}.Churn_Inference")
  time.sleep(5)
  refreshes = get_refreshes()

# Wait for refresh to finish
run_info = refreshes[0]
while run_info.state in (MonitorRefreshInfoState.PENDING, MonitorRefreshInfoState.RUNNING):
  run_info = w.quality_monitors.get_refresh(table_name=f"{catalog}.{shema}.Churn_Inference", refresh_id=run_info.refresh_id)
  print(f"waiting for refresh to complete {run_info.state}...")
  time.sleep(180)

assert run_info.state == MonitorRefreshInfoState.SUCCESS, "Monitor refresh failed"

In [0]:
w.quality_monitors.get(table_name=f"{catalog}.{schema}.Churn_Inference")

In [0]:
# w.quality_monitors.delete(table_name=f"{catalog}.{db}.advanced_churn_offline_inference", purge_artifacts=True)

## Inspeccionar dashboard
Ahora puede consultar el monitor dashboard generado. Navigue a la tabla de inferncia en **Catalog Explorer**, seleccione la ficha **Quality** y clic en el botón **View dashboar**.
Puede ver el número de inferencias realizadas antes del primer refresh del monitor.
Baje a la sección **Predict dift**, para ver la matriz de confusión y el % de predicción del modelo.
El siguiente paso es simular alguna desviación y refrescar el monitor de nuevo contra los nuevo datos capturados.